## PaDiM

In [ ]:
# IMPORTS
import json
import optuna
import torch
import warnings
import gc

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')

from anomalib.engine import Engine
import torch
from torchvision import models
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
import json
from anomalib.models.image.padim import Padim
import gc
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize
import optuna

# CUSTOM METRICS
class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

seed_everything(42, workers=True)

# DATA
# Необходимо указать путь к своим данным
train_dataset = Folder(
        name="main",
        root="/wrk/main/data",
        normal_dir="train",
        abnormal_dir="defect_test",
        #mask_dir="masks",
        normal_test_dir="normal_test",
        train_batch_size=10,
        eval_batch_size=10,
        num_workers=4,
        seed=42
    )

# CUSTOM CNN WEIGHTS
# Путь к кастомному feature extractor
weights = torch.load("/wrk/CNN_weights/resnet18.pth", map_location='cuda')
custom_backbone = models.resnet18()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

# OPTUNA
def objective(trial):
    layer_comb = [["layer1"], ["layer2"], ["layer3"], ["layer4"], ["layer1", "layer2"], ["layer2", "layer3"], ["layer3", "layer4"], ["layer1", "layer2", "layer3"], ["layer2", "layer3", "layer4"], ["layer1", "layer2", "layer3", "layer4"]]
    layers = trial.suggest_categorical("layers", layer_comb)
    
    n_features = trial.suggest_int("n_features", 10, 500, step=10)
    
    sensitivity = trial.suggest_float("sensitivity", 0.25, 0.65)
    
    resize_size = trial.suggest_categorical("resize_size", [256, 384, 512])
    
    pre_processor = PreProcessor(transform=Resize(size=(resize_size,resize_size)))
    
    post_processor = PostProcessor(
        image_sensitivity=sensitivity,
        pixel_sensitivity=sensitivity
    )
    
    evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC]) 
   
    try:
        model = Padim(
                backbone=custom_backbone,
                layers=layers,
                n_features=n_features,
                pre_trained=False,
                post_processor=post_processor,
                pre_processor=pre_processor,
                evaluator=evaluator)
        
        device = 'cuda'
        model = model.to(device)
        
        engine = Engine(accelerator='cuda', 
                        enable_progress_bar=True)
        
        engine.train(model=model, datamodule=train_dataset)
        
        metrics = engine.trainer.callback_metrics
        score = metrics["image_F1Score"].item()
        
        metrics = {k: v.item() for k, v in metrics.items()}
        
        log[trial.number] = {
            "trial": trial.number,
            "params": trial.params,
            "metrics": metrics,
            "result": score
        }

        with open("optuna_padim_18.json", "w") as f:
            json.dump(log, f, indent=4)

        return score

    except Exception as e:
        trial.set_user_attr("error", str(e))
        print(e)
        raise optuna.TrialPruned()

    finally:
        if "model" in locals():
            del model
        if "engine" in locals():
            del engine
        if "evaluator" in locals():
            del evaluator
        gc.collect()
        torch.cuda.empty_cache()

log = {}
study_18 = optuna.create_study(direction="maximize")
study_18.optimize(objective, n_trials=200)

best_params = study_18.best_params
best_score = study_18.best_value

print(best_params)
print(best_score)
    

In [ ]:
import json
import optuna
import torch
import warnings
import gc

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')

from anomalib.engine import Engine
import torch
from torchvision import models
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
import json
from anomalib.models.image.padim import Padim
import gc
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize
import optuna

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

seed_everything(42, workers=True)

train_dataset = Folder(
        name="main",
        root="/wrk/main/data",
        normal_dir="train",
        abnormal_dir="defect_test",
        #mask_dir="masks",
        normal_test_dir="normal_test",
        train_batch_size=10,
        eval_batch_size=10,
        num_workers=4,
        seed=42
    )

weights = torch.load("/wrk/CNN_weights/wide_resnet50_2.pth", map_location='cuda')
custom_backbone = models.wide_resnet50_2()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

def objective(trial):
    layer_comb = [["layer3"], ["layer4"], ["layer1", "layer2"], ["layer2", "layer3"], ["layer3", "layer4"], ["layer2", "layer3", "layer4"], ["layer1", "layer2", "layer3", "layer4"]]
    layers = trial.suggest_categorical("layers", layer_comb)
    
    n_features = trial.suggest_int("n_features", 10, 500, step=10)
    
    sensitivity = trial.suggest_float("sensitivity", 0.25, 0.65)
    
    resize_size = trial.suggest_categorical("resize_size", [256, 384, 512])
    
    pre_processor = PreProcessor(transform=Resize(size=(resize_size,resize_size)))
    
    post_processor = PostProcessor(
        image_sensitivity=sensitivity,
        pixel_sensitivity=sensitivity
    )
    
    evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC]) 
   
    try:
        model = Padim(
                backbone=custom_backbone,
                layers=layers,
                n_features=n_features,
                pre_trained=False,
                post_processor=post_processor,
                pre_processor=pre_processor,
                evaluator=evaluator)
        
        device = 'cuda'
        model = model.to(device)
        
        engine = Engine(accelerator='cuda', 
                        enable_progress_bar=True)
        
        engine.train(model=model, datamodule=train_dataset)
        
        metrics = engine.trainer.callback_metrics
        score = metrics["image_F1Score"].item()
        
        metrics = {k: v.item() for k, v in metrics.items()}
        
        log[trial.number] = {
            "trial": trial.number,
            "params": trial.params,
            "metrics": metrics,
            "result": score
        }

        with open("optuna_padim_50.json", "w") as f:
            json.dump(log, f, indent=4)
        
        return score

    except Exception as e:
        trial.set_user_attr("error", str(e))
        print(e)
        raise optuna.TrialPruned()

    finally:
        if "model" in locals():
            del model
        if "engine" in locals():
            del engine
        if "evaluator" in locals():
            del evaluator
        gc.collect()
        torch.cuda.empty_cache()

log = {}
study_50 = optuna.create_study(direction="maximize")
study_50.optimize(objective, n_trials=200)

best_params = study_50.best_params
best_score = study_50.best_value

print(best_params)
print(best_score)
    

## PatchCore

In [ ]:
import json
import optuna
import torch
import warnings
import gc

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')

from anomalib.engine import Engine
import torch
from torchvision import models
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
import json
from anomalib.models import Patchcore
import gc
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize
import optuna

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

seed_everything(42, workers=True)

train_dataset = Folder(
        name="main",
        root="/wrk/main/data",
        normal_dir="train",
        abnormal_dir="defect_test",
        #mask_dir="masks",
        normal_test_dir="normal_test",
        train_batch_size=10,
        eval_batch_size=10,
        num_workers=4,
        seed=42
    )

weights = torch.load("/wrk/CNN_weights/resnet18.pth", map_location='cuda')
custom_backbone = models.resnet18()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

def objective(trial):
    layer_comb = [["layer1"], ["layer2"], ["layer3"], ["layer4"], ["layer1", "layer2"], ["layer2", "layer3"], ["layer3", "layer4"], ["layer1", "layer2", "layer3"], ["layer2", "layer3", "layer4"], ["layer1", "layer2", "layer3", "layer4"]]
    layers = trial.suggest_categorical("layers", layer_comb)
    
    sensitivity = trial.suggest_float("sensitivity", 0.25, 0.65)
    
    sampling_ratios = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
    coreset_sampling_ratio = trial.suggest_categorical("coreset_sampling_ratio", sampling_ratios)
    
    neighbors = [1, 3, 5, 7, 9]
    num_neighbors = trial.suggest_categorical("num_neighbors", neighbors)
    
    resize_size = trial.suggest_categorical("resize_size", [256, 384, 512])
    
    pre_processor = PreProcessor(transform=Resize(size=(resize_size,resize_size)))
    
    post_processor = PostProcessor(
        image_sensitivity=sensitivity,
        pixel_sensitivity=sensitivity
    )
    
    evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC]) 
   
    try:
        model = Patchcore(
                backbone=custom_backbone,
                layers=layers,
                num_neighbors=num_neighbors,
                coreset_sampling_ratio=coreset_sampling_ratio,
                pre_trained=False,
                post_processor=post_processor,
                pre_processor=pre_processor,
                evaluator=evaluator)
        
        device = 'cuda'
        model = model.to(device)
        
        engine = Engine(accelerator='cuda', 
                        enable_progress_bar=True)
        
        engine.train(model=model, datamodule=train_dataset)
        
        metrics = engine.trainer.callback_metrics
        score = metrics["image_F1Score"].item()
        
        metrics = {k: v.item() for k, v in metrics.items()}
        
        log[trial.number] = {
            "trial": trial.number,
            "params": trial.params,
            "metrics": metrics,
            "result": score
        }

        with open("optuna_patchcore_18.json", "w") as f:
            json.dump(log, f, indent=4)

        return score

    except Exception as e:
        trial.set_user_attr("error", str(e))
        print(e)
        raise optuna.TrialPruned()

    finally:
        if "model" in locals():
            del model
        if "engine" in locals():
            del engine
        if "evaluator" in locals():
            del evaluator
        gc.collect()
        torch.cuda.empty_cache()

log = {}
study_18 = optuna.create_study(direction="maximize")
study_18.optimize(objective, n_trials=200)

best_params = study_18.best_params
best_score = study_18.best_value

print(best_params)
print(best_score)

In [ ]:
import json
import optuna
import torch
import warnings
import gc

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')

from anomalib.engine import Engine
import torch
from torchvision import models
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
import json
from anomalib.models import Patchcore
import gc
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize
import optuna

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

seed_everything(42, workers=True)

train_dataset = Folder(
        name="main",
        root="/wrk/main/data",
        normal_dir="train",
        abnormal_dir="defect_test",
        #mask_dir="masks",
        normal_test_dir="normal_test",
        train_batch_size=10,
        eval_batch_size=10,
        num_workers=4,
        seed=42
    )

weights = torch.load("/wrk/CNN_weights/wide_resnet50_2.pth", map_location='cuda')
custom_backbone = models.wide_resnet50_2()
custom_backbone.load_state_dict(weights)
custom_backbone.eval()

def objective(trial):
    layer_comb = [["layer1"], ["layer2"], ["layer3"], ["layer4"], ["layer1", "layer2"], ["layer2", "layer3"], ["layer3", "layer4"], ["layer1", "layer2", "layer3"], ["layer2", "layer3", "layer4"], ["layer1", "layer2", "layer3", "layer4"]]
    layers = trial.suggest_categorical("layers", layer_comb)
    
    sensitivity = trial.suggest_float("sensitivity", 0.25, 0.65)
    
    sampling_ratios = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
    coreset_sampling_ratio = trial.suggest_categorical("coreset_sampling_ratio", sampling_ratios)
    
    neighbors = [1, 3, 5, 7, 9]
    num_neighbors = trial.suggest_categorical("num_neighbors", neighbors)
    
    resize_size = trial.suggest_categorical("resize_size", [256, 384, 512])
    
    pre_processor = PreProcessor(transform=Resize(size=(resize_size,resize_size)))
    
    post_processor = PostProcessor(
        image_sensitivity=sensitivity,
        pixel_sensitivity=sensitivity
    )
    
    evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC]) 
   
    try:
        model = Patchcore(
                backbone=custom_backbone,
                layers=layers,
                num_neighbors=num_neighbors,
                coreset_sampling_ratio=coreset_sampling_ratio,
                pre_trained=False,
                post_processor=post_processor,
                pre_processor=pre_processor,
                evaluator=evaluator)
        
        device = 'cuda'
        model = model.to(device)
        
        engine = Engine(accelerator='cuda', 
                        enable_progress_bar=True)
        
        engine.train(model=model, datamodule=train_dataset)
        
        metrics = engine.trainer.callback_metrics
        score = metrics["image_F1Score"].item()
        
        metrics = {k: v.item() for k, v in metrics.items()}
        
        log[trial.number] = {
            "trial": trial.number,
            "params": trial.params,
            "metrics": metrics,
            "result": score
        }

        with open("optuna_patchcore_50.json", "w") as f:
            json.dump(log, f, indent=4)

        return score

    except Exception as e:
        trial.set_user_attr("error", str(e))
        print(e)
        raise optuna.TrialPruned()

    finally:
        if "model" in locals():
            del model
        if "engine" in locals():
            del engine
        if "evaluator" in locals():
            del evaluator
        gc.collect()
        torch.cuda.empty_cache()

log = {}
study_50 = optuna.create_study(direction="maximize")
study_50.optimize(objective, n_trials=200)

best_params = study_50.best_params
best_score = study_50.best_value

print(best_params)
print(best_score)
    

## Draem

In [ ]:
import json
import optuna
import torch
import warnings
import gc

warnings.filterwarnings("ignore")
torch.cuda.empty_cache()
torch.set_float32_matmul_precision('medium')

from anomalib.engine import Engine
import torch
from anomalib.data import Folder
from pytorch_lightning import seed_everything
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score, BinaryAUROC
from anomalib.metrics import Evaluator, AnomalibMetric
from anomalib.post_processing import PostProcessor
import json
from anomalib.models import Draem
import gc
from anomalib.pre_processing import PreProcessor
from torchvision.transforms.v2 import Resize
import optuna

class Accuracy(AnomalibMetric, BinaryAccuracy):
    pass

class Precision(AnomalibMetric, BinaryPrecision):
    pass

class Recall(AnomalibMetric, BinaryRecall):
    pass

class F1Score(AnomalibMetric, BinaryF1Score):
    pass

class AUROC(AnomalibMetric, BinaryAUROC):
    pass

image_Accuracy = Accuracy(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Precision = Precision(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_Recall = Recall(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_F1Score = F1Score(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

image_AUROC = AUROC(
    fields=["pred_label", "gt_label"],
    prefix="image_"
)

seed_everything(42, workers=True)

train_dataset = Folder(
        name="main",
        root="/wrk/main/data",
        normal_dir="train",
        abnormal_dir="defect_test",
        #mask_dir="masks",
        normal_test_dir="normal_test",
        train_batch_size=10,
        eval_batch_size=10,
        num_workers=4,
        seed=42
    )

def objective(trial):
    sensitivity = trial.suggest_float("sensitivity", 0.25, 0.65)
    
    beta = trial.suggest_float("beta", 0.1, 0.95)
    
    enable_sspcab = trial.suggest_categorical("enable_sspcab", [True, False])
    
    lambdas = [0.05, 0.1, 0.2]
    sspcab_lambda = trial.suggest_categorical("sspcab_lambda", lambdas)
    
    pre_processor = PreProcessor(transform=Resize(size=(resize_size,resize_size)))
    
    post_processor = PostProcessor(
        image_sensitivity=sensitivity,
        pixel_sensitivity=sensitivity
    )
    
    evaluator = Evaluator(test_metrics=[image_Accuracy, image_Precision, image_Recall, image_F1Score, image_AUROC]) 
   
    try:
        model = Draem(
            enable_sspcab=enable_sspcab,
            beta=beta,
            sspcab_lambda=sspcab_lambda,
            pre_processor=pre_processor,
            post_processor=post_processor,
            evaluator=evaluator)
        
        device = 'cuda'
        model = model.to(device)
        
        engine = Engine(accelerator='cuda', 
                        enable_progress_bar=True,
                        max_epochs=100)
        
        engine.train(model=model, datamodule=train_dataset)
        
        metrics = engine.trainer.callback_metrics
        score = metrics["image_F1Score"].item()
        
        metrics = {k: v.item() for k, v in metrics.items()}
        
        log[trial.number] = {
            "trial": trial.number,
            "params": trial.params,
            "metrics": metrics,
            "result": score
        }

        with open("optuna_draem.json", "w") as f:
            json.dump(log, f, indent=4)

        return score

    except Exception as e:
        trial.set_user_attr("error", str(e))
        print(e)
        raise optuna.TrialPruned()

    finally:
        if "model" in locals():
            del model
        if "engine" in locals():
            del engine
        if "evaluator" in locals():
            del evaluator
        gc.collect()
        torch.cuda.empty_cache()

log = {}
study_d = optuna.create_study(direction="maximize")
study_d.optimize(objective, n_trials=200)

best_params = study_d.best_params
best_score = study_d.best_value

print(best_params)
print(best_score)
    